In [0]:
from datetime import datetime

In [0]:
files = dbutils.fs.ls("s3://insurence-2025/RAW_INSURENCE_INDIAN_DATA/")
display(files)
today = datetime.today().strftime('%Y%m%d')
print(today)
filename = f"financial_{today}.csv"
print(filename)

In [0]:
if filename.endswith('.csv') and today in filename:
    # Run the next cell logic here
    file_path = f"s3://insurence-2025/RAW_INSURENCE_INDIAN_DATA/financial_{today}.csv"
    df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_path)
else:
    raise ValueError("Filename validation failed: Must be a .csv file and contain today's date.")

In [0]:
required_columns = [
    "queryid", "rptno", "TransactionType", "ForwardType", "SettlementType",
    "PositionDate", "Entity", "EntityGroup1", "EntityGroup2",
    "Counterparty", "CounterpartyGroup1", "CounterpartyGroup2",
    "Instrument", "InstrumentGroup1"
]

df_columns = df.columns
print("Columns in file:", df_columns)

missing_columns = [col for col in required_columns if col not in df_columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")
# Continue with next process if no error

In [0]:
file_path = f"s3://insurence-2025/RAW_INSURENCE_INDIAN_DATA/financial_{today}.csv"
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_path)

In [0]:
from pyspark.sql.functions import lit, current_timestamp

df_RAW = df.withColumn("filename", lit(filename)) \
       .withColumn("timestamp", lit(today))




In [0]:
from pyspark.sql.functions import (
    col, trim, upper, to_date, expr
)

df_silver = df_RAW.select(
    col("queryid").alias("query_id"),
    col("rptno").alias("report_no"),

    upper(trim(col("TransactionType"))).alias("transaction_type"),
    upper(trim(col("ForwardType"))).alias("forward_type"),
    upper(trim(col("SettlementType"))).alias("settlement_type"),

    expr("try_to_date(PositionDate, 'yyyy-MM-dd')").alias("position_date"),

    upper(trim(col("Entity"))).alias("entity"),
    upper(trim(col("EntityGroup1"))).alias("entity_group_1"),
    upper(trim(col("EntityGroup2"))).alias("entity_group_2"),

    upper(trim(col("Counterparty"))).alias("counterparty"),
    upper(trim(col("CounterpartyGroup1"))).alias("counterparty_group_1"),
    upper(trim(col("CounterpartyGroup2"))).alias("counterparty_group_2"),

    upper(trim(col("Instrument"))).alias("instrument"),
    upper(trim(col("InstrumentGroup1"))).alias("instrument_group_1"),

    upper(trim(col("CcyPair"))).alias("ccy_pair"),

    col("DealNo").alias("deal_no"),
    col("DealId").alias("deal_id"),
    col("DealSide").alias("deal_side"),

    upper(trim(col("PayorReceive"))).alias("pay_or_receive"),

    expr("try_to_date(CashFlowDate, 'yyyy-MM-dd')").alias("cashflow_date"),

    upper(trim(col("CashflowType"))).alias("cashflow_type"),
    upper(trim(col("Currency"))).alias("currency"),
    upper(trim(col("BaseCcy"))).alias("base_currency"),

    col("BaseForwardMTM").alias("base_forward_mtm"),
    col("BaseForwardNPVMTM").alias("base_forward_npv_mtm"),
    col("BaseNPVSpotMTM").alias("base_npv_spot_mtm"),

    col("FaceValue").alias("face_value"),
    col("PrincipalOutstanding").alias("principal_outstanding"),

    col("BPDelta").alias("bp_delta"),
    col("BPGamma").alias("bp_gamma"),
    col("IRR").alias("irr"),
    col("MacaulayDuration").alias("macaulay_duration"),
    col("ModifiedDuration").alias("modified_duration"),
    col("Theta").alias("theta"),
    col("Convexity").alias("convexity"),

    col("SpotFactor").alias("spot_factor"),
    col("SpotRate").alias("spot_rate"),
    col("ZeroRate").alias("zero_rate"),
    col("DiscountFactor").alias("discount_factor"),
    col("CurrencyDF").alias("currency_df"),

    col("FwdRate").alias("forward_rate"),
    col("FwdFactor").alias("forward_factor"),

    upper(trim(col("DiscountYieldCurve"))).alias("discount_yield_curve"),

    expr("try_to_date(DealDate, 'yyyy-MM-dd')").alias("deal_date"),

    col("MaturityDate").alias("maturity_date"),
    col("Term").alias("term"),

    col("filename").alias("source_file"),
    col("timestamp").alias("source_timestamp")
)
df_grouped = df_silver.groupBy("query_id").count()
display(df_grouped)
display(df_silver)

In [0]:
df_silver.write.mode("overwrite").format("csv").option("header", "true").save(f"s3://insurence-2025/CUREATED_INDIAN_DATA/finanace_{today}")
display(df_silver)